# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import uuid
from datetime import datetime
from typing import List, Dict, Optional, Literal, TypedDict

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run

In [3]:
# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(usecwd=True), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Environment variables loaded.")

Environment variables loaded.


In [4]:
# Model used by the agent and by the LLM-as-judge evaluation tool
MODEL_NAME = "gpt-4o-mini"

# Minimum judge confidence needed to trust internal knowledge without searching the web
CONFIDENCE_THRESHOLD = 0.7


def short(value, limit: int = 600) -> str:
    """Compact, truncated string version of a tool result for printing."""
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)
    return text if len(text) <= limit else text[:limit] + " ...[truncated]"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# Same path, collection name and embedding function as Part 1
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=OPENAI_BASE_URL,
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print(f"Loaded collection '{collection.name}' with {collection.count()} games")


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    - Genre and Publisher of the game
    - distance: semantic distance to the query (lower means more similar)
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["metadatas", "distances"],
    )
    docs = []
    for doc_id, meta, dist in zip(results["ids"][0], results["metadatas"][0], results["distances"][0]):
        docs.append({
            "id": doc_id,
            "Name": meta["Name"],
            "Platform": meta["Platform"],
            "YearOfRelease": meta["YearOfRelease"],
            "Genre": meta["Genre"],
            "Publisher": meta["Publisher"],
            "Description": meta["Description"],
            "distance": round(dist, 4),
        })
    return docs


# Quick test of the tool
for d in retrieve_game(query="first 3D platformer Mario game")[:3]:
    print(d["Name"], d["Platform"], d["YearOfRelease"], d["distance"])

Loaded collection 'udaplay' with 15 games


Super Mario 64 Nintendo 64 1996 0.1042
Super Mario World Super Nintendo Entertainment System (SNES) 1990 0.1272
Mario Kart 8 Deluxe Nintendo Switch 2017 0.1862


#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation about the evaluation result")
    confidence: float = Field(description="Confidence from 0.0 to 1.0 that the documents fully answer the question")


evaluation_parser = PydanticOutputParser(model_class=EvaluationReport)


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
      (pass each document as a JSON string)
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    - confidence: 0.0-1.0 confidence that the documents fully answer the question
    """
    judge = LLM(model=MODEL_NAME, temperature=0.0)
    docs_text = "\n".join(f"- {doc}" for doc in retrieved_docs) or "(no documents)"
    messages = [
        SystemMessage(content=(
            "You are an expert judge for a video game knowledge base. "
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not. "
            "Mark the documents as useful ONLY if they contain the specific facts needed to answer "
            "the question (e.g., the exact game, platform, year or publisher asked about). "
            "Documents about other, merely similar games are NOT useful. "
            "Questions about current events, news, ongoing projects or anything after the documents' "
            "information are NOT answerable from them. "
            "Also give a confidence between 0.0 and 1.0 that the documents fully answer the question."
        )),
        UserMessage(content=f"User question: {question}\n\nRetrieved documents:\n{docs_text}"),
    ]
    response = judge.invoke(messages, response_format=EvaluationReport)
    report: EvaluationReport = evaluation_parser.parse(response)
    return report.model_dump()


# Quick test of the tool
print(evaluate_retrieval(
    question="Was Mortal Kombat X released for PlayStation 5?",
    retrieved_docs=[json.dumps(d) for d in retrieve_game(query="Mortal Kombat X PlayStation 5")],
))

{'useful': False, 'description': "The retrieved documents do not contain any information about Mortal Kombat X or its release on PlayStation 5. They only mention other games, none of which are related to Mortal Kombat X. Therefore, they do not provide the specific facts needed to answer the user's question regarding the release of Mortal Kombat X on PlayStation 5.", 'confidence': 0.0}


#### Game Web Search Tool

In [7]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the internet (Tavily API) for game industry information
    that is not available in the internal vector DB.
    args:
    - question: a question about game industry.

    You'll receive a dict with:
    - answer: a short summary answer generated from the web results
    - results: list of web pages, each with title, url and content
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": (r.get("content") or "")[:700],
            }
            for r in response.get("results", [])
        ],
    }


# Quick test of the tool
test = game_web_search(question="Was Mortal Kombat X released for PlayStation 5?")
print(test["answer"])
print([r["url"] for r in test["results"]])

Mortal Kombat X was originally released for PlayStation 4, Xbox One, and PC on April 14, 2015, and was never released natively for PlayStation 5. However, it is playable on PS5 through the PlayStation Plus Collection, which includes it as part of its launch lineup. To play the game on PS5, your system may need to be updated to the latest system software, and some features available on PS4 may be absent. The game is also available for PS4 and PS5 as part of the PlayStation Plus lineup.
['https://www.facebook.com/fb-answers/mortal-kombat-x-playstation-5-release', 'https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000', 'https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V', 'https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved', 'https://www.ign.com/games/mortal-kombat-x']


### Agent

In [8]:
UDAPLAY_INSTRUCTIONS = f"""
You are UdaPlay, an AI research assistant for the video game industry.
You answer questions about games: titles, release dates, platforms, genres, descriptions and publishers.

You MUST follow this workflow for every question about games, in this exact order, one tool at a time:
1. RETRIEVE: ALWAYS start by calling `retrieve_game` with a standalone search query - even when you expect
   the local database not to have the answer (e.g. news or current events). Never skip this step. If the user asks a follow-up
   (e.g. "who published it?"), use the conversation history to rewrite it with the full game name.
2. EVALUATE: call `evaluate_retrieval` with the user's question and the retrieved documents
   (each document as a JSON string).
3. FALLBACK: if the evaluation says `useful` is false OR `confidence` < {CONFIDENCE_THRESHOLD},
   call `game_web_search` with the question. Otherwise do NOT search the web.
4. ANSWER: write the final answer using ONLY information from the tool results.
   Combine internal and web information when both are relevant. Never invent facts.

This applies to ALL questions about games, studios, publishers, platforms or industry news, including
questions like "What is <company> working on right now?" - retrieve and evaluate first, then search the web.
The ONLY exception: questions that are only about this conversation itself (e.g. "which of the games I asked about is the oldest?")
can be answered directly from the conversation history. For comparisons, first list every game with its
release year from the history, then compare the years carefully (smaller year = released earlier) before concluding.

Format the final answer in Markdown exactly like this:
**Answer:** <one or two sentence direct answer>
**Details:**
- <supporting facts: platform, year, publisher, genre... as relevant>
**Confidence:** <High | Medium | Low> - <short reason>
**Sources:**
- Internal knowledge base: <game name(s) from the vector DB> (only if used)
- Web: <page title> - <url> (one line per web page used; always include the URL)
"""

udaplay_agent = Agent(
    model_name=MODEL_NAME,
    instructions=UDAPLAY_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)
print("UdaPlay agent ready with tools:", [t.name for t in udaplay_agent.tools])

UdaPlay agent ready with tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


In [9]:
from IPython.display import Markdown, display


def print_trace(run: Run):
    """Show the steps the agent took for the latest question: tool calls, tool results, final answer."""
    state = run.get_final_state()
    messages = state["messages"]
    last_user_idx = max(i for i, m in enumerate(messages) if m.role == "user")
    step = 0
    print("\n" + "=" * 100)
    for m in messages[last_user_idx:]:
        if m.role == "user":
            print(f"USER QUESTION: {m.content}")
        elif m.role == "assistant" and m.tool_calls:
            for call in m.tool_calls:
                step += 1
                print(f"\n[Step {step}] TOOL CALL -> {call.function.name}({short(call.function.arguments, 300)})")
        elif m.role == "tool":
            print(f"   RESULT <- {m.name}: {short(json.loads(m.content), 600)}")
    tools_used = [m.name for m in messages[last_user_idx:] if m.role == "tool"]
    print(f"\nWorkflow: {' -> '.join(tools_used + ['final answer'])}")
    print(f"Web fallback used: {'game_web_search' in tools_used} | total tokens so far: {state.get('total_tokens')}")
    print("=" * 100)
    display(Markdown(messages[-1].content))


def ask(question: str, session_id: str = "udaplay_demo"):
    run = udaplay_agent.invoke(question, session_id=session_id)
    print_trace(run)
    return run


# Query 1 - answerable from the local dataset
_ = ask("When was Pokémon Gold and Silver released?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: When was Pokémon Gold and Silver released?

[Step 1] TOOL CALL -> retrieve_game({"query":"Pokémon Gold and Silver release date"})
   RESULT <- retrieve_game: [{'id': '006', 'Name': 'Pokémon Gold and Silver', 'Platform': 'Game Boy Color', 'YearOfRelease': 1999, 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'distance': 0.1266}, {'id': '007', 'Name': 'Pokémon Ruby and Sapphire', 'Platform': 'Game Boy Advance', 'YearOfRelease': 2002, 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.', 'distance': 0.1608}, {'id': '012', 'Name': 'Ma ...[truncated]

[Step 2] TOOL CALL -> evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":["{\"

**Answer:** Pokémon Gold and Silver were released in 1999.  
**Details:**
- Platform: Game Boy Color
- Genre: Role-playing
- Publisher: Nintendo  
**Confidence:** High - The retrieved document provides the exact release year and relevant details.  
**Sources:**
- Internal knowledge base: Pokémon Gold and Silver

In [10]:
# Query 2 - answerable from the local dataset
_ = ask("Which one was the first 3D platformer Mario game?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: Which one was the first 3D platformer Mario game?

[Step 1] TOOL CALL -> retrieve_game({"query":"first 3D platformer Mario game"})
   RESULT <- retrieve_game: [{'id': '009', 'Name': 'Super Mario 64', 'Platform': 'Nintendo 64', 'YearOfRelease': 1996, 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'distance': 0.1042}, {'id': '008', 'Name': 'Super Mario World', 'Platform': 'Super Nintendo Entertainment System (SNES)', 'YearOfRelease': 1990, 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'di ...[truncated]

[Step 2] TOOL CALL -> evaluate_retrieval({"question":"Which one was the first 3D platformer Mario game?","retrieved_doc

**Answer:** The first 3D platformer Mario game is Super Mario 64.  
**Details:**
- Platform: Nintendo 64
- Year of Release: 1996
- Publisher: Nintendo  
**Confidence:** High - The retrieved document provides specific information confirming Super Mario 64 as the first 3D platformer.  
**Sources:**
- Internal knowledge base: Super Mario 64

In [11]:
# Query 3 - NOT in the local dataset -> web search fallback
_ = ask("Was Mortal Kombat X realeased for Playstation 5?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: Was Mortal Kombat X realeased for Playstation 5?

[Step 1] TOOL CALL -> retrieve_game({"query":"Mortal Kombat X release for Playstation 5"})
   RESULT <- retrieve_game: [{'id': '005', 'Name': "Marvel's Spider-Man 2", 'Platform': 'PlayStation 5', 'YearOfRelease': 2023, 'Genre': 'Action-adventure', 'Publisher': 'Sony Interactive Entertainment', 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'distance': 0.1914}, {'id': '015', 'Name': 'Halo Infinite', 'Platform': 'Xbox Series X|S', 'YearOfRelease': 2021, 'Genre': 'First-person shooter', 'Publisher': 'Xbox Game Studios', 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-wo ...[truncated]

[Step 2] TOOL CALL -> evaluate_retrieval({"question":"Was Mortal Kombat X released for Playstation 5?","retri

**Answer:** Mortal Kombat X was not released natively for PlayStation 5, but it is playable on the console through the PlayStation Plus Collection.  
**Details:**
- Original Release Date: April 14, 2015
- Platforms: Originally released for PlayStation 4, Xbox One, and PC
- PlayStation Plus Collection: Included as part of the PS5 launch lineup  
**Confidence:** High - The information confirms the game's availability on PS5 through the PlayStation Plus Collection.  
**Sources:**
- Mortal Kombat X - PlayStation Store - https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000
- PS5 PlayStation Plus Collection To Include Mortal Kombat X - Mortal Kombat Online - https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V

In [12]:
# Query 4 - current events, can't be answered from the dataset -> web search fallback
_ = ask("What is Rockstar Games working on right now?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: What is Rockstar Games working on right now?

[Step 1] TOOL CALL -> retrieve_game({"query":"Rockstar Games current projects"})
   RESULT <- retrieve_game: [{'id': '002', 'Name': 'Grand Theft Auto: San Andreas', 'Platform': 'PlayStation 2', 'YearOfRelease': 2004, 'Genre': 'Action-adventure', 'Publisher': 'Rockstar Games', 'Description': "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.", 'distance': 0.1913}, {'id': '003', 'Name': 'Gran Turismo 5', 'Platform': 'PlayStation 3', 'YearOfRelease': 2010, 'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving p ...[truncated]

[Step 2] TOOL CALL -> evaluate_retrieval({"question":"What is Rockstar Games working on right now?","retrieved_docs":["{\"i

**Answer:** Rockstar Games is currently working on Grand Theft Auto VI (GTA VI), which is set to release on November 19, 2026.  
**Details:**
- Development: The game has been in development for some time and faced a data breach in 2022.
- Acquisitions: Rockstar has also acquired Cfx.re and Video Games Deluxe, and rebranded Ruffian Games as Rockstar Dundee.  
**Confidence:** High - The information is sourced from multiple credible updates regarding Rockstar's current projects.  
**Sources:**
- What is Rockstar currently working on - Reddit - https://www.reddit.com/r/rockstar/comments/kzwi63/what_is_rockstar_currently_working_on
- Every Upcoming Rockstar Games Title in Development - IMDb - https://www.imdb.com/news/ni65706242
- Rockstar Games - Wikipedia - https://en.wikipedia.org/wiki/Rockstar_Games

In [13]:
# Query 5 - follow-up that relies on conversation state ("it" = Super Mario 64 from query 2)
_ = ask("Going back to the first 3D Mario game you mentioned: who published it and what genre is it?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: Going back to the first 3D Mario game you mentioned: who published it and what genre is it?

[Step 1] TOOL CALL -> retrieve_game({"query":"Super Mario 64 publisher and genre"})
   RESULT <- retrieve_game: [{'id': '009', 'Name': 'Super Mario 64', 'Platform': 'Nintendo 64', 'YearOfRelease': 1996, 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'distance': 0.1022}, {'id': '008', 'Name': 'Super Mario World', 'Platform': 'Super Nintendo Entertainment System (SNES)', 'YearOfRelease': 1990, 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'di ...[truncated]

[Step 2] TOOL CALL -> evaluate_retrieval({"question":"who published Super

**Answer:** Super Mario 64 was published by Nintendo and belongs to the platformer genre.  
**Details:**
- Publisher: Nintendo
- Genre: Platformer  
**Confidence:** High - The retrieved document provides clear and direct information regarding the publisher and genre.  
**Sources:**
- Internal knowledge base: Super Mario 64

In [14]:
# Query 6 - question about the session itself (short-term memory)
_ = ask("Of all the games I asked about in this conversation, which one was released first?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: Of all the games I asked about in this conversation, which one was released first?

Workflow: final answer
Web fallback used: False | total tokens so far: 7164


Here are the games you asked about along with their release years:

1. **Super Mario 64** - 1996
2. **Pokémon Gold and Silver** - 1999
3. **Mortal Kombat X** - 2015
4. **Grand Theft Auto VI** - Set to release in 2026

Based on these years, **Super Mario 64** was released first in 1996.

In [15]:
# The conversation state is kept per session: every run of the session is stored in memory
runs = udaplay_agent.get_session_runs("udaplay_demo")
print(f"Runs stored in session 'udaplay_demo': {len(runs)}")
print(f"Messages in the conversation: {len(runs[-1].get_final_state()['messages'])}")

# A different session starts with a clean history
_ = ask("Which games did I ask about before?", session_id="another_session")

Runs stored in session 'udaplay_demo': 6
Messages in the conversation: 37
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

USER QUESTION: Which games did I ask about before?

Workflow: final answer
Web fallback used: False | total tokens so far: 851


You have not asked about any games in this conversation yet.

### (Optional) Advanced

In [16]:
# Advanced UdaPlay:
#   1. Long-term memory: what the agent learns from web searches is stored in a persistent
#      ChromaDB collection, so later sessions can answer from it without searching the web again.
#   2. The workflow is an explicit state machine where every tool is a pre-defined node:
#      rewrite_query -> retrieve -> evaluate -> (web_search) -> generate_answer -> (memorize)
#   3. Structured output: every answer is returned in natural language AND as JSON (AgentReport).


# ---------- 1. Long-term memory ----------
class LongTermGameMemory:
    """Persistent memory of facts learned from the web, stored next to the game collection."""

    def __init__(self, client, name: str = "udaplay_long_term_memory", max_distance: float = 0.35):
        self.client = client
        self.name = name
        self.max_distance = max_distance
        self.collection = client.get_or_create_collection(name, embedding_function=embedding_fn)

    def reset(self):
        self.client.delete_collection(self.name)
        self.collection = self.client.get_or_create_collection(self.name, embedding_function=embedding_fn)

    def remember(self, question: str, answer: str, sources: List[str]):
        self.collection.add(
            ids=[str(uuid.uuid4())],
            documents=[f"Q: {question}\nA: {answer}"],
            metadatas=[{
                "question": question,
                "sources": ", ".join(sources),
                "timestamp": int(datetime.now().timestamp()),
            }],
        )

    def recall(self, query: str, k: int = 3) -> List[dict]:
        if self.collection.count() == 0:
            return []
        res = self.collection.query(query_texts=[query], n_results=min(k, self.collection.count()),
                                    include=["documents", "metadatas", "distances"])
        return [
            {"memory": doc, "sources": meta["sources"], "distance": round(dist, 4)}
            for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
            if dist <= self.max_distance
        ]


# ---------- 3. Structured output ----------
class Source(BaseModel):
    type: Literal["internal_db", "long_term_memory", "web"]
    reference: str = Field(description="Game name for internal_db, or the URL for web/long_term_memory")


class AgentReport(BaseModel):
    answer: str = Field(description="Natural language answer to the question")
    key_facts: List[str] = Field(description="Short supporting facts")
    games: List[str] = Field(description="Names of the games the answer is about")
    confidence: Literal["high", "medium", "low"]
    used_web_search: bool
    sources: List[Source]


class QueryRewrite(BaseModel):
    standalone_question: str


# ---------- 2. State machine with tools as nodes ----------
class UdaPlayState(TypedDict):
    question: str
    session_id: str
    history: List[dict]
    search_query: str
    retrieved_docs: List[dict]
    memory_hits: List[dict]
    evaluation: dict
    web_results: Optional[dict]
    report: Optional[dict]
    total_tokens: int


class UdaPlayWorkflowAgent:
    def __init__(self, model_name: str = MODEL_NAME, long_term_memory: LongTermGameMemory = None):
        self.model_name = model_name
        self.long_term_memory = long_term_memory
        self.sessions: Dict[str, List[dict]] = {}   # short-term memory: Q/A history per session
        self.workflow = self._create_state_machine()

    def _llm(self, messages, response_format, state):
        response = LLM(model=self.model_name, temperature=0.0).invoke(messages, response_format=response_format)
        tokens = state.get("total_tokens", 0) + (response.token_usage.total_tokens if response.token_usage else 0)
        return response_format.model_validate_json(response.content), tokens

    # ---- nodes ----
    def _rewrite_query(self, state: UdaPlayState) -> UdaPlayState:
        if not state["history"]:
            print(f"  [rewrite_query] no history, using question as is")
            return {"search_query": state["question"]}
        history = "\n".join(f"Q: {h['question']}\nA: {h['answer']}" for h in state["history"][-5:])
        result, tokens = self._llm([
            SystemMessage(content="Rewrite the user's latest question as a standalone search query about video games, "
                                  "resolving pronouns using the conversation history. Return only the rewritten question."),
            UserMessage(content=f"Conversation history:\n{history}\n\nLatest question: {state['question']}"),
        ], QueryRewrite, state)
        print(f"  [rewrite_query] '{state['question']}' -> '{result.standalone_question}'")
        return {"search_query": result.standalone_question, "total_tokens": tokens}

    def _retrieve(self, state: UdaPlayState) -> UdaPlayState:
        docs = retrieve_game(query=state["search_query"])
        memories = self.long_term_memory.recall(state["search_query"]) if self.long_term_memory else []
        print(f"  [retrieve] vector DB top-3: {[(d['Name'], d['distance']) for d in docs[:3]]}")
        print(f"  [retrieve] long-term memory hits: {len(memories)}")
        return {"retrieved_docs": docs, "memory_hits": memories}

    def _evaluate(self, state: UdaPlayState) -> UdaPlayState:
        docs = [json.dumps(d, ensure_ascii=False) for d in state["retrieved_docs"]]
        docs += [json.dumps(m, ensure_ascii=False) for m in state["memory_hits"]]
        evaluation = evaluate_retrieval(question=state["search_query"], retrieved_docs=docs)
        print(f"  [evaluate] useful={evaluation['useful']} confidence={evaluation['confidence']:.2f}")
        print(f"             {short(evaluation['description'], 300)}")
        return {"evaluation": evaluation}

    def _web_search(self, state: UdaPlayState) -> UdaPlayState:
        results = game_web_search(question=state["search_query"])
        print(f"  [web_search] {len(results['results'])} pages: {[r['url'] for r in results['results']]}")
        return {"web_results": results}

    def _generate_answer(self, state: UdaPlayState) -> UdaPlayState:
        context = {
            "internal_db_documents": state["retrieved_docs"] if state["evaluation"]["useful"] else [],
            "long_term_memory": state["memory_hits"],
            "web_results": state["web_results"],
            "evaluation": state["evaluation"],
        }
        history = "\n".join(f"Q: {h['question']}\nA: {h['answer']}" for h in state["history"][-5:])
        report, tokens = self._llm([
            SystemMessage(content="You are UdaPlay, a video game research assistant. Answer using ONLY the provided "
                                  "context and conversation history. Cite every source you used: internal_db with the "
                                  "game name, web or long_term_memory with the URL. Set used_web_search to true only "
                                  "if web_results were used."),
            UserMessage(content=f"Conversation history:\n{history or '(none)'}\n\nQuestion: {state['question']}\n\n"
                                f"Context:\n{json.dumps(context, ensure_ascii=False)}"),
        ], AgentReport, state)
        print(f"  [generate_answer] confidence={report.confidence} sources={len(report.sources)}")
        return {"report": report.model_dump(), "total_tokens": tokens}

    def _memorize(self, state: UdaPlayState) -> UdaPlayState:
        report = state["report"]
        urls = [s["reference"] for s in report["sources"] if s["type"] == "web"]
        self.long_term_memory.remember(state["search_query"], report["answer"], urls)
        print(f"  [memorize] stored web finding in long-term memory ({self.long_term_memory.collection.count()} memories)")
        return {}

    def _create_state_machine(self) -> StateMachine[UdaPlayState]:
        machine = StateMachine[UdaPlayState](UdaPlayState)
        entry = EntryPoint[UdaPlayState]()
        rewrite = Step[UdaPlayState]("rewrite_query", self._rewrite_query)
        retrieve = Step[UdaPlayState]("retrieve_game", self._retrieve)
        evaluate = Step[UdaPlayState]("evaluate_retrieval", self._evaluate)
        web = Step[UdaPlayState]("game_web_search", self._web_search)
        answer = Step[UdaPlayState]("generate_answer", self._generate_answer)
        memorize = Step[UdaPlayState]("memorize", self._memorize)
        termination = Termination[UdaPlayState]()
        machine.add_steps([entry, rewrite, retrieve, evaluate, web, answer, memorize, termination])

        machine.connect(entry, rewrite)
        machine.connect(rewrite, retrieve)
        machine.connect(retrieve, evaluate)

        def good_enough(state: UdaPlayState):
            ev = state["evaluation"]
            return answer if ev["useful"] and ev["confidence"] >= CONFIDENCE_THRESHOLD else web

        machine.connect(evaluate, [answer, web], good_enough)
        machine.connect(web, answer)

        def should_memorize(state: UdaPlayState):
            return memorize if (state.get("web_results") and self.long_term_memory) else termination

        machine.connect(answer, [memorize, termination], should_memorize)
        machine.connect(memorize, termination)
        return machine

    def invoke(self, question: str, session_id: str = "default") -> AgentReport:
        history = self.sessions.setdefault(session_id, [])
        print(f"\n{'=' * 100}\nUSER QUESTION ({session_id}): {question}")
        run = self.workflow.run({
            "question": question, "session_id": session_id, "history": list(history),
            "search_query": question, "retrieved_docs": [], "memory_hits": [],
            "evaluation": {}, "web_results": None, "report": None, "total_tokens": 0,
        })
        state = run.get_final_state()
        report = AgentReport(**state["report"])
        history.append({"question": question, "answer": report.answer})

        path = " -> ".join(s.step_id for s in run.snapshots if s.step_id != "__entry__")
        print(f"\nWorkflow path: {path} | tokens: {state['total_tokens']}")
        sources_md = "\n".join(f"- {s.type}: {s.reference}" for s in report.sources)
        facts_md = "\n".join(f"- {f}" for f in report.key_facts)
        display(Markdown(f"**Answer:** {report.answer}\n\n**Details:**\n{facts_md}\n\n"
                         f"**Confidence:** {report.confidence}\n\n**Sources:**\n{sources_md}"))
        print("Structured JSON output:")
        print(report.model_dump_json(indent=2))
        return report


long_term_memory = LongTermGameMemory(chroma_client)
long_term_memory.reset()   # start the demo with an empty memory so the "learning" is visible
advanced_agent = UdaPlayWorkflowAgent(long_term_memory=long_term_memory)
print("Advanced agent ready. Long-term memories:", long_term_memory.collection.count())

Advanced agent ready. Long-term memories: 0


In [17]:
# Session 1: in the dataset -> answered from internal knowledge, no web search
_ = advanced_agent.invoke("Who published Grand Theft Auto: San Andreas?", session_id="session_1")


USER QUESTION (session_1): Who published Grand Theft Auto: San Andreas?
[StateMachine] Starting: __entry__
  [rewrite_query] no history, using question as is
[StateMachine] Executing step: rewrite_query


  [retrieve] vector DB top-3: [('Grand Theft Auto: San Andreas', 0.1243), ('Gran Turismo', 0.211), ('Gran Turismo 5', 0.2162)]
  [retrieve] long-term memory hits: 0
[StateMachine] Executing step: retrieve_game


  [evaluate] useful=True confidence=1.00
             The retrieved document with ID '002' provides the necessary information to answer the user's question about who published 'Grand Theft Auto: San Andreas'. It explicitly states that the publisher is 'Rockstar Games'. The other documents do not pertain to 'Grand Theft Auto: San Andreas' and therefore  ...[truncated]
[StateMachine] Executing step: evaluate_retrieval


  [generate_answer] confidence=high sources=1
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__

Workflow path: rewrite_query -> retrieve_game -> evaluate_retrieval -> generate_answer | tokens: 926


**Answer:** Grand Theft Auto: San Andreas was published by Rockstar Games.

**Details:**
- Published by Rockstar Games
- Released in 2004
- Genre: Action-adventure

**Confidence:** high

**Sources:**
- internal_db: Grand Theft Auto: San Andreas

Structured JSON output:
{
  "answer": "Grand Theft Auto: San Andreas was published by Rockstar Games.",
  "key_facts": [
    "Published by Rockstar Games",
    "Released in 2004",
    "Genre: Action-adventure"
  ],
  "games": [
    "Grand Theft Auto: San Andreas"
  ],
  "confidence": "high",
  "used_web_search": false,
  "sources": [
    {
      "type": "internal_db",
      "reference": "Grand Theft Auto: San Andreas"
    }
  ]
}


In [18]:
# Session 1: not in the dataset -> web search, and the finding is stored in long-term memory
_ = advanced_agent.invoke("Was Mortal Kombat X released for PlayStation 5?", session_id="session_1")


USER QUESTION (session_1): Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Starting: __entry__


  [rewrite_query] 'Was Mortal Kombat X released for PlayStation 5?' -> 'Was Mortal Kombat X released for PlayStation 5?'
[StateMachine] Executing step: rewrite_query


  [retrieve] vector DB top-3: [("Marvel's Spider-Man 2", 0.2101), ("Marvel's Spider-Man", 0.226), ('Halo Infinite', 0.2268)]
  [retrieve] long-term memory hits: 0
[StateMachine] Executing step: retrieve_game


  [evaluate] useful=False confidence=0.00
             The retrieved documents do not contain any information about Mortal Kombat X, its release platforms, or its release date. They only mention other games, specifically Marvel's Spider-Man 2, Marvel's Spider-Man, Halo Infinite, Gran Turismo 5, and Kinect Adventures!, none of which are related to Mortal ...[truncated]
[StateMachine] Executing step: evaluate_retrieval
  [web_search] 5 pages: ['https://www.facebook.com/fb-answers/mortal-kombat-x-playstation-5-release', 'https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000', 'https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V', 'https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved', 'https://www.ign.com/games/mortal-kombat-x']
[StateMachine] Executing step: game_web_search


  [generate_answer] confidence=high sources=3
[StateMachine] Executing step: generate_answer


  [memorize] stored web finding in long-term memory (1 memories)
[StateMachine] Executing step: memorize
[StateMachine] Terminating: __termination__

Workflow path: rewrite_query -> retrieve_game -> evaluate_retrieval -> game_web_search -> generate_answer -> memorize | tokens: 2013


**Answer:** Mortal Kombat X was not released natively for PlayStation 5. It was originally released for PlayStation 4, Xbox One, and PC on April 14, 2015. However, it is playable on PS5 through the PlayStation Plus Collection, which includes it as part of its launch lineup.

**Details:**
- Mortal Kombat X was released on April 14, 2015.
- It is playable on PS5 via the PlayStation Plus Collection.
- There is no native PS5 version of Mortal Kombat X.

**Confidence:** high

**Sources:**
- web: https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V
- web: https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000
- web: https://www.facebook.com/fb-answers/mortal-kombat-x-playstation-5-release

Structured JSON output:
{
  "answer": "Mortal Kombat X was not released natively for PlayStation 5. It was originally released for PlayStation 4, Xbox One, and PC on April 14, 2015. However, it is playable on PS5 through the PlayStation Plus Collection, which includes it as part of its launch lineup.",
  "key_facts": [
    "Mortal Kombat X was released on April 14, 2015.",
    "It is playable on PS5 via the PlayStation Plus Collection.",
    "There is no native PS5 version of Mortal Kombat X."
  ],
  "games": [
    "Mortal Kombat X"
  ],
  "confidence": "high",
  "used_web_search": true,
  "sources": [
    {
      "type": "web",
      "reference": "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V"
    },
    {
      "type": "web",
      "reference": "https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000"
    },
    {
      "type": "web",
      "reference": "https://www.facebook.com/fb-answers/mor

In [19]:
# Session 1: follow-up resolved with the conversation history ("it" = Mortal Kombat X)
_ = advanced_agent.invoke("Who developed it?", session_id="session_1")


USER QUESTION (session_1): Who developed it?
[StateMachine] Starting: __entry__


  [rewrite_query] 'Who developed it?' -> 'Who developed Mortal Kombat X?'
[StateMachine] Executing step: rewrite_query


  [retrieve] vector DB top-3: [('Super Smash Bros. Melee', 0.2107), ('Halo Infinite', 0.2269), ('Kinect Adventures!', 0.2315)]
  [retrieve] long-term memory hits: 1
[StateMachine] Executing step: retrieve_game


  [evaluate] useful=False confidence=0.00
             The retrieved documents do not contain any information about the developer of Mortal Kombat X. They focus on other games, such as Super Smash Bros. Melee, Halo Infinite, and others, which are unrelated to Mortal Kombat X. The only relevant memory mentions the release platforms and dates for Mortal K ...[truncated]
[StateMachine] Executing step: evaluate_retrieval
  [web_search] 5 pages: ['https://en.wikipedia.org/wiki/Mortal_Kombat_X', 'https://www.instagram.com/p/DecS5qEidyR', 'https://mortalkombat.fandom.com/wiki/Mortal_Kombat_X', 'https://www.ign.com/games/mortal-kombat-x', 'https://store.steampowered.com/app/307780/Mortal_Kombat_X']
[StateMachine] Executing step: game_web_search


  [generate_answer] confidence=high sources=2
[StateMachine] Executing step: generate_answer


  [memorize] stored web finding in long-term memory (2 memories)
[StateMachine] Executing step: memorize
[StateMachine] Terminating: __termination__

Workflow path: rewrite_query -> retrieve_game -> evaluate_retrieval -> game_web_search -> generate_answer -> memorize | tokens: 2085


**Answer:** Mortal Kombat X was developed by NetherRealm Studios and published by Warner Bros. Interactive Entertainment. Additionally, High Voltage Software developed the Windows version of the game, with Polish studio QLOC taking over the work on it shortly after the release of Kombat Pack 1.

**Details:**
- Developed by NetherRealm Studios
- Published by Warner Bros. Interactive Entertainment
- Windows version developed by High Voltage Software and QLOC

**Confidence:** high

**Sources:**
- web: https://en.wikipedia.org/wiki/Mortal_Kombat_X
- web: https://mortalkombat.fandom.com/wiki/Mortal_Kombat_X

Structured JSON output:
{
  "answer": "Mortal Kombat X was developed by NetherRealm Studios and published by Warner Bros. Interactive Entertainment. Additionally, High Voltage Software developed the Windows version of the game, with Polish studio QLOC taking over the work on it shortly after the release of Kombat Pack 1.",
  "key_facts": [
    "Developed by NetherRealm Studios",
    "Published by Warner Bros. Interactive Entertainment",
    "Windows version developed by High Voltage Software and QLOC"
  ],
  "games": [
    "Mortal Kombat X"
  ],
  "confidence": "high",
  "used_web_search": true,
  "sources": [
    {
      "type": "web",
      "reference": "https://en.wikipedia.org/wiki/Mortal_Kombat_X"
    },
    {
      "type": "web",
      "reference": "https://mortalkombat.fandom.com/wiki/Mortal_Kombat_X"
    }
  ]
}


In [20]:
# Session 2 (new conversation, empty short-term history):
# the agent now answers from long-term memory, learned in session 1, without searching the web again
_ = advanced_agent.invoke("Can I play Mortal Kombat X on a PS5?", session_id="session_2")


USER QUESTION (session_2): Can I play Mortal Kombat X on a PS5?
[StateMachine] Starting: __entry__
  [rewrite_query] no history, using question as is
[StateMachine] Executing step: rewrite_query


  [retrieve] vector DB top-3: [("Marvel's Spider-Man 2", 0.221), ("Marvel's Spider-Man", 0.2385), ('Halo Infinite', 0.244)]
  [retrieve] long-term memory hits: 2
[StateMachine] Executing step: retrieve_game


  [evaluate] useful=True confidence=0.90
             The retrieved documents contain specific information regarding Mortal Kombat X, including its original release platform (PlayStation 4) and the fact that it is playable on PlayStation 5 through the PlayStation Plus Collection. This directly answers the user's question about whether they can play Mor ...[truncated]
[StateMachine] Executing step: evaluate_retrieval


  [generate_answer] confidence=high sources=3
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__

Workflow path: rewrite_query -> retrieve_game -> evaluate_retrieval -> generate_answer | tokens: 1361


**Answer:** Yes, you can play Mortal Kombat X on a PS5. Although it was not released natively for the PS5, it is available through the PlayStation Plus Collection, which includes it as part of its launch lineup.

**Details:**
- Mortal Kombat X was originally released for PlayStation 4.
- It is playable on PS5 via the PlayStation Plus Collection.

**Confidence:** high

**Sources:**
- long_term_memory: https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V
- long_term_memory: https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000
- long_term_memory: https://www.facebook.com/fb-answers/mortal-kombat-x-playstation-5-release

Structured JSON output:
{
  "answer": "Yes, you can play Mortal Kombat X on a PS5. Although it was not released natively for the PS5, it is available through the PlayStation Plus Collection, which includes it as part of its launch lineup.",
  "key_facts": [
    "Mortal Kombat X was originally released for PlayStation 4.",
    "It is playable on PS5 via the PlayStation Plus Collection."
  ],
  "games": [
    "Mortal Kombat X"
  ],
  "confidence": "high",
  "used_web_search": false,
  "sources": [
    {
      "type": "long_term_memory",
      "reference": "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V"
    },
    {
      "type": "long_term_memory",
      "reference": "https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000"
    },
    {
      "type": "long_term_memory",
      "reference": "https://www.facebook.com/fb-answers/mortal-kombat-x-playstation-5-release"
    }
  ]
}


In [21]:
# What the agent has learned so far
learned = long_term_memory.collection.get(include=["documents", "metadatas"])
for doc, meta in zip(learned["documents"], learned["metadatas"]):
    print(doc.replace("\n", " | "), "| sources:", meta["sources"][:200], "\n")

Q: Was Mortal Kombat X released for PlayStation 5? | A: Mortal Kombat X was not released natively for PlayStation 5. It was originally released for PlayStation 4, Xbox One, and PC on April 14, 2015. However, it is playable on PS5 through the PlayStation Plus Collection, which includes it as part of its launch lineup. | sources: https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V, https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000, http 

Q: Who developed Mortal Kombat X? | A: Mortal Kombat X was developed by NetherRealm Studios and published by Warner Bros. Interactive Entertainment. Additionally, High Voltage Software developed the Windows version of the game, with Polish studio QLOC taking over the work on it shortly after the release of Kombat Pack 1. | sources: https://en.wikipedia.org/wiki/Mortal_Kombat_X, https://mortalkombat.fandom.com/wiki/Mortal_Kombat_X 

